# 037_incremental_merge
Silver load for the high-volume feeds (ledger lines, till sales).

1. Take the Bronze rows newer than the Silver table's watermark (`_batch_id`), as `{{increment}}`.
2. Run the flow's script over just those rows.
3. `MERGE` the result into Silver on the flow's key: existing keys are updated, new ones inserted.
   Running the same batch twice changes nothing.

First run, or `full_refresh=True`: the whole Bronze table is processed and the Silver table
replaced.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

## Loader

In [ ]:
def silver_incremental_merge(config, batch_id):
    increment, is_full_load = prepare_increment(config)
    df = with_lineage(run_script(config), batch_id)
    schema, table_name = config["destination_schema"], config["destination_table"]
    if is_full_load:
        return write_table(df, "silver", schema, table_name)
    if increment.limit(1).count() == 0:
        print(f"  {table_name}: nothing new")
        return 0
    return merge_table(df, "silver", schema, table_name, config["primary_key_columns"])


LOADERS["037_incremental_merge"] = silver_incremental_merge

## Run on its own

In [ ]:
# Parameters: set a control_id and run all to load one flow on its own (handy for testing).
# Left at 0, this notebook only defines its loader for the orchestrator.
control_id = 0
batch_id = ""

In [ ]:
if control_id:
    print(execute_flow(control_id, batch_id or new_batch_id(), silver_incremental_merge, triggered_by="manual"))